In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-04-20 05:43:26--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.004s  

2026-04-20 05:43:26 (52.4 MB/s) - ‘names.txt’ saved [228145/228145]



In [ ]:
stoi = {chr(i): i - ord('a') + 1 for i in range(ord('a'), ord('z') + 1)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

In [ ]:
words = open('names.txt').read().splitlines()

In [ ]:
X, Y = [], []

cnt = 3
dim = 10

for w in words:
  context = [0] * cnt
  for lt in w + '.':
    X.append(context)
    Y.append(stoi[lt])
    context = context[1:] + [stoi[lt]]

X = torch.tensor(X)
Y = torch.tensor(Y)
len(X)

228146

In [ ]:
C = torch.randn((27, dim))

In [ ]:
emb = C[X]
emb = emb.view(emb.shape[0], dim * cnt)
emb.shape

torch.Size([228146, 30])

In [ ]:
W1 = torch.randn((dim * cnt, 100))
b1 = torch.randn(100)
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
params = [W1, b1, W2, b2]

for p in params:
  p.requires_grad = True

In [ ]:
ix = torch.randint(0, X.shape[0], (32,))
print(C[X[ix]].shape)
emb = C[X[ix]].view(32, dim * cnt)

torch.Size([32, 3, 10])


In [ ]:
for k in range(200000):
  #выбрала случайную группу данных
  ix = torch.randint(0, X.shape[0], (32,))
  emb = C[X[ix]].view(-1, dim * cnt)

  #forward pass
  h = torch.tanh(emb @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y[ix])

  #backward pass
  for p in params:
    p.grad = None

  loss.backward()

  #update
  for p in params:
    p.data += -0.1 * p.grad
  if k % 10000 == 0:
    print(loss.item())

16.02528190612793
2.8907086849212646
1.9711874723434448
2.4115047454833984
2.6817548274993896
2.4524362087249756
2.29154372215271
2.3207950592041016
2.1535520553588867
2.1381356716156006
2.412543535232544
2.1777336597442627
2.4154536724090576
2.0345418453216553
2.3953893184661865
2.5271708965301514
2.0672831535339355
2.336944341659546
2.2703096866607666
2.509230375289917


In [ ]:
C[X].shape

In [ ]:
emb = C[X]
h = torch.tanh(emb.view(-1, dim * cnt) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

In [ ]:
for i in range(20):
  context = [0] * cnt
  ans = ""
  while True:
    h = torch.tanh(C[context].view(dim * cnt) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits)
    ix = torch.multinomial(probs, num_samples=1).item()
    if ix == 0:
      break
    ans += itos[ix]
    context = context[1:] + [ix]
  print(ans)